In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install keras-tuner

In [ ]:
 #import libraries
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping, ModelCheckpoint
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import os
import keras_tuner
from keras_tuner.tuners import RandomSearch

# Data augmentation
# https://numpy.org/doc/stable/reference/routines.array-manipulation.html
directory="/content/drive/My Drive/Day6/animal_data"
folder_names=["Bear","Bird","Cat","Cow","Deer","Dog","Dolphin","Elephant","Giraffe","Horse","Kangaroo","Lion","Panda","Tiger","Zebra"]
print(folder_names,"\n",len(folder_names))

['Bear', 'Bird', 'Cat', 'Cow', 'Deer', 'Dog', 'Dolphin', 'Elephant', 'Giraffe', 'Horse', 'Kangaroo', 'Lion', 'Panda', 'Tiger', 'Zebra'] 
 15


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator


In [ ]:
datagen = ImageDataGenerator(
        rotation_range=0.5,  # Randomly rotate images in the range
        zoom_range = 0.2, # Randomly zoom image
        width_shift_range=0.1,  # Randomly shift images horizontally
        height_shift_range=0.1,  # Randomly shift images vertically
)
datagen.fit(X_train)

NameError: name 'X_train' is not defined

In [ ]:
#from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [ ]:
import cv2

def resize_image(image):
    new_height=new_width=224
    return cv2.resize(image,(new_height,new_width))


# augment function
def augment_image(image):
    augmented_images = []

    # Random horizontal flip
    if np.random.rand() < 0.6:
        augmented_images.append(cv2.flip(image, 1))

    # Random brightness adjustment
    brightness_factor = np.random.uniform(0.9, 1.1)
    augmented_images.append(np.clip(image * brightness_factor, 0, 255).astype(np.uint8))

    # Random contrast adjustment
    contrast_factor = np.random.uniform(0.1, 0.2)
    augmented_images.append(np.clip(image * contrast_factor, 0, 255).astype(np.uint8))

    # Random rotation (90, 180, or 270 degrees)
    rotation_angle = np.random.choice([0, 90, 180, 270])
    if rotation_angle != 0:
        augmented_images.append(np.rot90(image, k=rotation_angle // 90))   # นำมุมที่ random มาหาร 90 แล้วปัดลง

    return augmented_images



# Data augmentation function applied to each of original image
for folder_name in os.listdir(directory):
    folder_path = os.path.join(directory, folder_name)

    # Check if the item in the directory is a folder
    if os.path.isdir(folder_path):
        print(f"Augmenting images in folder: {folder_name}")

        # Iterate over each image file in the subfolder
        for filename in os.listdir(folder_path):
            if filename.endswith((".jpeg", ".jpg")):  # Check if the file is an image file

                image_path = os.path.join(folder_path, filename)

                # Read the image using OpenCV
                image = cv2.imread(image_path)
                # Check if the image is successfully loaded
                if image is not None:

                    #first resize the image and replace with the original one
                    image=resize_image(image)
                    cv2.imwrite(image_path,image)

                    # Augment the image
                    augmented_images = augment_image(image)

                    # Save augmented images in the same folder
                    for i, augmented_image in enumerate(augmented_images):
                        new_filename = f"{filename.split('.')[0]}_{i+1}.jpg"
                        new_image_path = os.path.join(folder_path, new_filename)
                        cv2.imwrite(new_image_path, augmented_image)
                else:
                    print(f"Failed to load image: {image_path}")


Augmenting images in folder: Bird
Augmenting images in folder: Cat
Augmenting images in folder: Elephant
Augmenting images in folder: Kangaroo
Augmenting images in folder: Dog
Augmenting images in folder: Zebra
Augmenting images in folder: Tiger
Augmenting images in folder: Dolphin
Augmenting images in folder: Horse
Augmenting images in folder: Giraffe
Augmenting images in folder: Panda
Augmenting images in folder: Lion
Augmenting images in folder: Cow
Augmenting images in folder: Bear
Augmenting images in folder: Deer


In [ ]:
#################################################################################
####################### After preapear the dataset, let's train #################
#################################################################################

In [ ]:
#import libraries
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping, ModelCheckpoint
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import os
import keras_tuner
from keras_tuner.tuners import RandomSearch

In [ ]:
# Loading dataset To work with image, tensorflow provides excellent libraries such as -
# Using Dataset from directory
# Image Data generator and flow from directory
# Here I will use the former process whcih is very easy to comprehend.
image_height=224
image_width=224
batch_size=16

ds_train=tf.keras.preprocessing.image_dataset_from_directory(
    directory,
    labels="inferred",
    label_mode="int",
    color_mode="rgb",
    batch_size=batch_size,
    image_size=(image_height,image_width),
    shuffle=True,
    seed=12,
    validation_split=.1,
    subset="training",

)

ds_validation=tf.keras.preprocessing.image_dataset_from_directory(
    directory,
    labels="inferred",
    label_mode="int",
    color_mode="rgb",
    batch_size=batch_size,
    image_size=(image_height,image_width),
    shuffle=True,
    seed=12,
    validation_split=.1,
    subset="validation"
)

# Normalization: Normalization of image refers to keeping the pixel values in between
# 0 and 1 or 1 and -1. Normalizing helps stablize and speed up the training process.

def normalize_img(image, label):
    # Normalize image
    normalized_image = tf.cast(image, tf.float32) / 255.0
    return normalized_image, label

In [ ]:
# Prepare dataset
AUTOTUNE = tf.data.experimental.AUTOTUNE
ds_train = ds_train.map(normalize_img, num_parallel_calls=AUTOTUNE)
ds_train = ds_train.cache()
ds_train = ds_train.prefetch(AUTOTUNE)
ds_validation=ds_validation.prefetch(AUTOTUNE)

# Now the data is loaded and preprocessed and ready for training. Now at first,
# I will create a custom model (sequential model) using keras API. As my dataset is small,
# I will create two layer (CNN layer) to keep the model simple for avoiding overfitting.
# I used 16 and 32 number of filters in the first and second layer (CNN layer) respectively.
model = keras.Sequential([
    layers.Input(shape=(224, 224, 3)),
    layers.Conv2D(16, 3, padding="same", name="conv1", kernel_regularizer=keras.regularizers.l2(.01)),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Conv2D(32, 3, padding="same", name="conv2", kernel_regularizer=keras.regularizers.l2(.01)),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dropout(.5),
    layers.Dense(15, name="output_layer"),
])

model.summary()

NameError: name 'ds_train' is not defined

In [ ]:
model.compile(
loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=keras.optimizers.Adam(learning_rate=.0001),
    metrics=["accuracy"],
)

model.fit(ds_train,epochs=10,batch_size=batch_size,validation_data=ds_validation,verbose=2)

Epoch 1/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 973s 2s/step - accuracy: 0.3675 - loss: 5.8213 - val_accuracy: 0.2662 - val_loss: 3183.0186
Epoch 2/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 950s 2s/step - accuracy: 0.6539 - loss: 2.2810 - val_accuracy: 0.4423 - val_loss: 2385.1919
Epoch 3/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 953s 2s/step - accuracy: 0.7681 - loss: 1.5054 - val_accuracy: 0.4930 - val_loss: 1927.4810
Epoch 4/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 908s 2s/step - accuracy: 0.8515 - loss: 1.0572 - val_accuracy: 0.4972 - val_loss: 2120.7827
Epoch 5/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 916s 2s/step - accuracy: 0.8701 - loss: 0.8520 - val_accuracy: 0.5915 - val_loss: 2026.0885
Epoch 6/10
322/400 ━━━━━━━━━━━━━━━━━━━━ 2:50 2s/step - accuracy: 0.9082 - loss: 0.6828